## Part 1: Getting familiar with Gymnasium

##### Copyright by UIT-NC@NT549

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>
</div>

## Part 1: Getting familiar with Gymnasium

## Part 2: Custom Environment "VacuumCleaner"

In [ ]:
"""
Suggested owner: Person A

Merge contract:
- Keep the class name `VacuumCleanerEnv`
- Keep methods `__init__`, `reset`, `step`, `render`
- Keep observation keys `position` and `dust`
- Keep action mapping `0=up, 1=down, 2=left, 3=right`
"""

# Build a simple custom Gymnasium environment named "VacuumCleaner-v0".
# The environment simulates a vacuum robot operating in an m x n room. The robot can
# move up, down, left, and right and automatically vacuums the cell it occupies.
# The objective is to clean all dust particles in the room. There is a single obstacle
# located at a specified cell (i, j) that the robot must avoid. Entering the obstacle
# cell yields a large negative reward and terminates the episode.
# The robot receives a positive reward when it vacuums a dirty cell. If the robot
# attempts to vacuum an already clean cell, that action receives a reduced reward
# (e.g., penalized or halved). When all dust has been cleaned, the agent receives
# a large positive bonus reward and the episode terminates.
# Action space: Discrete(4) -> {0: up, 1: down, 2: left, 3: right}
# Observation space: Dict with 'position' (x, y) and 'dust' grid (m x n binary)

import gymnasium as gym
import numpy as np
import os
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output

class VacuumCleanerEnv(gym.Env):
    def __init__(self, m=5, n=5, obstacle=(2, 2)):
        super(VacuumCleanerEnv, self).__init__()
        self.m = m
        self.n = n
        self.obstacle = tuple(obstacle)

        # Action space: 0=up, 1=down, 2=left, 3=right
        ### YOU NEED TO WRITE YOUR CODE BELOW ###

        self.action_space = gym.spaces.Discrete(4)  # 4 hành động: up, down, left, right

        # Observation space: position and dust grid
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        # HERE
        self.observation_space = gym.spaces.Dict({
            'position': gym.spaces.Box(
                low=np.array([0, 0]),           #Tọa độ lớn nhất là (m-1, n-1) và bé nhất là (0, 0)
                high=np.array([m-1, n-1]), 
                shape=(2,),                     #Vector 2 phần tử (x, y)
                dtype=np.int32                  #kiểu dữ liệu trong mảng Numpy
            )
                ,
            'dust': gym.spaces.Box(
                low=0,                          #0: sạch, 1: bẩn
                high=1, 
                shape=(m, n), 
                dtype=np.int32
            )
        })

        self.reset()

    def reset(self, *, seed=None, options=None):
        # initialize position and dust
        # Start the robot at the top-left corner (row 0, column 0)
        # Use NUMPY to define.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.position = np.array([0,0], dtype=np.int32)  # HERE

        # Initialize dust grid: 1 indicates dirty, 0 indicates clean.
        # Shape is (m, n) corresponding to the room dimensions.
        # Use NUMPY to define.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.dust_grid = np.random.randint(0,2,(self.m,self.n))  # HERE

        # Ensure the obstacle cell contains no dust (robot cannot clean there).
        # This also prevents rewarding the agent for occupying the obstacle.
        self.dust_grid[self.obstacle] = 0  # obstacle cell has no dust
        self.total_reward = 0.0
        self.truncated = False
        self.terminated = False
        obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
        return obs, {}

    def step(self, action):
        # compute candidate new position
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        if action == 0:   # Up
            candidate = self.position + np.array([-1,0])
        elif action == 1: # Down
            candidate = self.position + np.array([1,0])
        elif action == 2: # Left
            candidate = self.position + np.array([0,-1])
        elif action == 3: # Right
            candidate = self.position + np.array([0,1])
        else:
            candidate = self.position.copy()

        # boundary check
        if (0 <= candidate[0] < self.m) and (0 <= candidate[1] < self.n):
            # obstacle check
            if tuple(candidate) == self.obstacle:
                self.position = candidate.copy()
                reward = -10.0
                self.terminated = True
                obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
                self.total_reward += reward
                return obs, reward, True, False, {}
            else:
                self.position = candidate.copy()
        # else: stay in place
        x, y = self.position

        if self.dust_grid[x, y] == 1:
            reward = 1.0
            self.dust_grid[x, y] = 0
        else:
            reward = -0.5
        # If the robot is on a dirty cell, give a positive reward (1.0) and mark it clean.
        # If the cell is already clean, apply a small penalty (-0.5) to discourage redundant cleaning.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        
        self.total_reward += reward

        # check if all cleaned
        if np.sum(self.dust_grid) == 0:
            reward += 10.0
            self.terminated = True

        obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
        return obs, reward, bool(self.terminated), bool(self.truncated), {}

    def render(self, mode='human'):
        # In Jupyter notebooks, use IPython.display.clear_output to clear the cell output.
        try:
            clear_output(wait=True)
        except Exception:
            # Fallback for terminal execution
            os.system('cls' if os.name == 'nt' else 'clear')

        # Build display grid with symbols:
        # '#' obstacle, '.' dirty, ' ' clean, 'R' robot, 'X' robot on obstacle
        display = np.full((self.m, self.n), ' ', dtype='<U1')
        for i in range(self.m):
            for j in range(self.n):
                if (i, j) == self.obstacle:
                    display[i, j] = '#'
                elif self.dust_grid[i, j] == 1:
                    display[i, j] = '.'
                else:
                    display[i, j] = ' '

        x, y = int(self.position[0]), int(self.position[1])
        if (x, y) == self.obstacle:
            display[x, y] = 'X'
        else:
            display[x, y] = 'R'

        for row in display:
            print(''.join(row))
        print(f"Total reward: {self.total_reward}")
        time.sleep(0.15)


In [ ]:
def robot_policy(option, env=None):
     """
     A simple policy function that selects an action based on the specified option.
     Currently supports only a random policy.
     """
     if option == "random":
          return env.action_space.sample()  # Randomly select an action from the action space
     
     elif option == "round_robin":
          if not hasattr(robot_policy, "actions"):

               m, n = env.m, env.n
               actions = []

               for i in range(m):

                    if i % 2 == 0:  
                         actions += [3] * (n-1)   # move right
                    else:
                         actions += [2] * (n-1)   # move left

                    if i < m-1:
                         actions.append(1)        # move down

               robot_policy.actions = actions
               robot_policy.counter = 0

          action = robot_policy.actions[robot_policy.counter % len(robot_policy.actions)]
          robot_policy.counter += 1
          return action
          
     elif option == "priority_based":
          x, y = env.position
          dust = env.dust_grid

          moves = {
               0: (x-1, y),  # up
               1: (x+1, y),  # down
               2: (x, y-1),  # left
               3: (x, y+1)   # right
          }

        # ưu tiên đi tới ô có bụi
          for action, (nx, ny) in moves.items():
               if 0 <= nx < env.m and 0 <= ny < env.n:
                    if (nx, ny) != env.obstacle and dust[nx, ny] == 1:
                         return action

          # nếu không thấy bụi xung quanh → random
          return env.action_space.sample()
          
     

In [ ]:
def save(env, episode, filename="result.txt"):
    with open(filename, "a") as f:
        f.write(f"Episode {episode}: Total reward = {env.total_reward}\n")

In [ ]:
def plot_rewards(rewards):

    plt.figure(figsize=(8,5))

    plt.plot(rewards)

    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.title("Reward per Episode")

    plt.grid(True)

    plt.show()

In [ ]:
if __name__ == "__main__":
    
    rewards = []
    for episode in range(100:

        env = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2))
        obs, _ = env.reset()

        terminated = False
        truncated = False

        print(f"\n===== Episode {episode+1} =====")

        while not terminated and not truncated:
            action = robot_policy(option="round_robin", env=env)
            obs, reward, terminated, truncated, info = env.step(action)

            env.render()
            print(f"Episode {episode+1} - Action: {action}, Reward: {reward}, Terminated: {terminated}")
        rewards.append(env.total_reward)
        save(env, episode+1)
    plot_rewards(rewards)

Evaluation and Analysis
## CONGRATULATIONS TEAM!
Congratulations to the team for completing Part 2,3 of Lab01 - Introduction to Python, Gymnasium and Formulating RL Problem.
Keep up the effort in the next sections.
References: https://gymnasium.farama.org/ 

Suggested additional practice resources: https://gymnasium.farama.org/introduction/create_custom_env/
## ADDITIONAL INFORMATION
**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn
